# 06. Retrieval and explanation with LangChain

The models produce a ranked list. An investigator opening that list needs to know **why** a
provider is there, in plain language, with the billing codes decoded. That is a retrieval and
generation task, and this notebook builds it with LangChain so the pieces have the names the
job posting uses.

## What gets built

1. A **vector store** of every HCPCS procedure code and its description, embedded so they can
   be searched by meaning rather than by keyword.
2. A **retriever** over that store. Ask "long-acting injectable antipsychotic" and get back the
   J-codes for those drugs, even though none of those words appear in the code descriptions.
3. A **chain** that takes a provider NPI and a question, pulls the provider's billing facts from
   DuckDB, retrieves the relevant code knowledge, and has Claude write the explanation.
4. A **simple agent**: the same lookups exposed as tools that the model decides when to call.

## The vocabulary, mapped to what you already did by hand

The Colo project did retrieval-augmented generation with direct API calls. LangChain is the
same architecture with names for each piece. The translation:

| LangChain term | what it is | what it was in Colo |
|---|---|---|
| `Document` | a chunk of text plus a metadata dict | a dict with `text` and `source` |
| document loader | reads a source into Documents | the PubMed fetch and parse code |
| text splitter | cuts long Documents into chunks | the sentence-window chunker |
| `Embeddings` | text to vector | the sentence-transformers call |
| vector store | stores vectors, does nearest-neighbor search | ChromaDB, same library |
| `Retriever` | "give me the k most relevant Documents for this query" | the `collection.query()` wrapper |
| `PromptTemplate` | a prompt with `{placeholders}` | an f-string |
| chain / LCEL | pieces joined with `|` so output of one feeds the next | the function that called them in order |
| output parser | turns the model response into a string or object | `response.content[0].text` |
| tool | a Python function with a schema the model can ask to call | the tool definitions in the API call |
| agent | a loop where the model picks tools until it is done | the manual tool-use loop |

The framework buys you interchangeable parts and a common interface. It costs you a layer of
abstraction over calls you could make directly. Both are legitimate; knowing when each fits is
the interview answer.

## 0. Setup

**What this cell does:** imports, loads the Anthropic API key from a `.env` file (gitignored),
and opens the database read-only. If no key is found the notebook still runs everything up to
the model call and prints the prompt it would have sent.

In [1]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

PROJECT = "C:/Users/palla/OneDrive/Documents/Coding Projects/Medicare Fraud ML"
load_dotenv(f"{PROJECT}/.env")
HAVE_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("Anthropic key loaded:", HAVE_KEY)

con = duckdb.connect(f"{PROJECT}/database/medicare_fraud.duckdb", read_only=True)
con.execute("SET enable_progress_bar = false")
def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.tools import tool
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_chroma import Chroma
from langchain_anthropic import ChatAnthropic

Anthropic key loaded: True


C:\Users\palla\AppData\Local\Temp\ipykernel_41532\67935210.py:23: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import FastEmbedEmbeddings


## 1. Documents: every procedure code, once

**What this cell does:** pulls the distinct HCPCS codes and descriptions out of
`provider_service` and wraps each as a LangChain `Document`. The text is "code: description"
so a search can match on either. The metadata carries the code and whether it is a drug, so
results can be filtered or displayed without parsing the text back apart.

**Why no text splitter:** each description is one sentence. Splitting is for documents longer
than the embedding model's window (a few hundred words here). Knowing when you do not need a
step is part of knowing the step.

In [2]:
codes = q("""
SELECT HCPCS_Cd AS code,
       any_value(HCPCS_Desc) AS description,
       any_value(HCPCS_Drug_Ind) AS drug_ind,
       count(DISTINCT Rndrng_NPI) AS n_providers,
       sum(Tot_Srvcs * Avg_Mdcr_Pymt_Amt) AS total_paid
FROM provider_service
GROUP BY HCPCS_Cd
""")
docs = [
    Document(page_content=f"{r.code}: {r.description}",
             metadata={"code": r.code, "drug": r.drug_ind == "Y",
                       "n_providers": int(r.n_providers), "total_paid": float(r.total_paid)})
    for r in codes.itertuples()
]
print(f"{len(docs):,} documents")
docs[0]

6,471 documents


Document(metadata={'code': 'G0105', 'drug': False, 'n_providers': 8537, 'total_paid': 105008875.71001454}, page_content='G0105: Colorectal cancer screening; colonoscopy on individual at high risk')

## 2. Embeddings and the vector store

**What this cell does:** embeds every document with a small open model that runs on CPU
(`bge-small-en-v1.5`, via fastembed, no PyTorch needed) and stores the vectors in ChromaDB on
disk. If the store already exists it is opened instead of rebuilt, so this cell is cheap after
the first run.

**Why a persistent store:** the same reason the DuckDB file exists. Embed once, query many
times. The store lives under `database/` on the D: drive with the rest of the big files.

Chroma is the same library Colo uses. The only difference is that LangChain's wrapper gives it
the standard `Retriever` interface so it can be swapped for FAISS, pgvector, or anything else
without changing the chain.

In [3]:
CHROMA_DIR = f"{PROJECT}/database/chroma_hcpcs"
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

t0 = time.time()
if Path(CHROMA_DIR).exists() and any(Path(CHROMA_DIR).iterdir()):
    store = Chroma(collection_name="hcpcs", embedding_function=embeddings, persist_directory=CHROMA_DIR)
    print(f"opened existing store with {store._collection.count():,} vectors ({time.time() - t0:.1f}s)")
else:
    store = Chroma.from_documents(docs, embeddings, collection_name="hcpcs", persist_directory=CHROMA_DIR)
    print(f"built store with {store._collection.count():,} vectors ({time.time() - t0:.0f}s)")

opened existing store with 6,471 vectors (0.2s)


## 3. The retriever, and why "semantic" matters

**What this cell does:** wraps the store as a retriever that returns the five closest documents
and runs three queries. None of the queries share vocabulary with the code descriptions they
should find. That is the point of embeddings over keyword search: an investigator can ask in
plain English.

In [4]:
retriever = store.as_retriever(search_kwargs={"k": 5})

for query in ["long-acting injectable antipsychotic",
              "urine drug screening panel",
              "removing a suspicious mole in the office"]:
    print(f"\n== {query}")
    for d in retriever.invoke(query):
        print(f"   {d.page_content[:110]}")


== long-acting injectable antipsychotic
   90865: Injection of hypnotic drug for psychiatric diagnosis or therapy
   J1631: Injection, haloperidol decanoate, per 50 mg
   J2794: Injection, risperidone (risperdal consta), 0.5 mg
   J2426: Injection, paliperidone palmitate extended release (invega sustenna), 1 mg
   99152: Use of a drug to induce depression of consciousness by physician performing a procedure (5 years or old

== urine drug screening panel
   84431: Urine analysis for thromboxane (lipid)
   80069: Kidney function blood test panel
   81005: Analysis of urine, except immunoassays
   80076: Liver function blood test panel
   80418: Anterior pituitary gland evaluation panel

== removing a suspicious mole in the office
   54110: Removal of thickened tissue of penis
   65436: Removal of outer layer of cornea with application of chelating agent
   30300: Removal of foreign body in nose
   65222: Removal of foreign body in cornea using slit lamp
   15853: Removal of sutures or s

**Read the three results honestly.** The first query is a clean win: it returns the J-codes for
haloperidol decanoate, risperidone, and paliperidone palmitate, none of which share a word with
"long-acting injectable antipsychotic." The second and third are weak. "Urine drug screening"
pulls in blood panels and a thromboxane assay; "removing a suspicious mole" gets foreign-body
removals from the nose and cornea. The embedding model is a small general-purpose one, the
code descriptions are terse, and medical vocabulary is where small general models are thinnest.

That is a normal state for a first retrieval pass, and the fixes are standard: a
domain-trained embedding model (PubMedBERT-style, which the Colo project already uses), a
hybrid retriever that combines vector similarity with keyword matching, or a reranker on the
top 20. Being able to say which of those you would try first, and how you would measure the
improvement, is worth more than a demo that happened to work.

## 4. Provider context from DuckDB

The chain needs facts about the provider: who they are, what they bill most, and why the model
scored them. **What this cell does:** one function that assembles that from the database as a
short text block. This is the "augmented" part of retrieval-augmented generation: structured
data the model could never guess, handed to it as context.

In [5]:
FEATURES = ["srvcs_per_bene", "pymt_per_bene", "chrg_to_alowd", "drug_pymt_share", "risk_score",
            "pymt_per_bene_per_risk", "pct_under65", "pct_dual", "avg_age", "n_distinct_codes",
            "tot_benes", "sameday_repeat_ratio", "em_high_share", "top_code_pymt_share",
            "facility_share", "n_service_rows"]

def provider_context(npi: int) -> str:
    p = q(f"""
        SELECT f.provider_type, f.state, f.entity_code, f.label, f.n_extreme,
               g.gbm_score, g.gbm_calibrated, s.iforest_score,
               {", ".join("f." + c for c in FEATURES)},
               {", ".join("f.z_" + c for c in FEATURES)}
        FROM provider_features f
        LEFT JOIN scores_gbm g USING (npi)
        LEFT JOIN scores_iforest s USING (npi)
        WHERE f.npi = {npi}
    """).iloc[0]
    top_codes = q(f"""
        SELECT HCPCS_Cd AS code, HCPCS_Desc AS description, Place_Of_Srvc AS place,
               Tot_Benes AS benes, Tot_Srvcs AS services,
               round(Tot_Srvcs * Avg_Mdcr_Pymt_Amt) AS paid
        FROM provider_service WHERE Rndrng_NPI = {npi}
        ORDER BY paid DESC LIMIT 8
    """)
    total_paid = top_codes["paid"].sum() if len(top_codes) else 0
    z = pd.Series({c: p[f"z_{c}"] for c in FEATURES}).dropna()
    drivers = z.abs().sort_values(ascending=False).head(5)

    lines = [
        f"Provider NPI {npi}: {p['provider_type']}, {p['state']}, "
        f"{'individual' if p['entity_code'] == 'I' else 'organization'}.",
        f"Model rank: gradient-boosting score {p['gbm_score']:.3f} "
        f"(calibrated probability {p['gbm_calibrated']:.4f}), "
        f"isolation-forest anomaly score {p['iforest_score']:.3f}, "
        f"{int(p['n_extreme'])} features beyond 3 peer spreads.",
        "Peer-adjusted z-scores driving the rank (positive = above peers of the same specialty):",
    ]
    lines += [f"  {c}: z = {z[c]:+.2f}  (raw value {p[c]:.3g})" for c in drivers.index]
    if len(top_codes) == 0:
        lines.append("Top billed codes in 2024: NONE PUBLISHED. CMS suppresses every code-level row "
                     "with fewer than 11 patients, so this provider's service mix is not visible "
                     "in the public file.")
    else:
        lines.append("Top billed codes in 2024 by Medicare payment (this provider's actual billing):")
        for r in top_codes.itertuples():
            lines.append(f"  {r.code} ({'facility' if r.place == 'F' else 'office'}): {r.description[:80]}"
                         f" | {int(r.benes)} patients, {int(r.services)} services, ${int(r.paid):,}"
                         f" ({100 * r.paid / total_paid:.0f}% of payments)")
    return "\n".join(lines)

# pick the two providers to explain: the top out-of-fold score among providers whose code-level
# rows are published (so there is billing to describe), and the highest-ranked known positive
top_npi = int(q("""
SELECT f.npi FROM provider_features f JOIN scores_gbm g USING (npi)
WHERE f.label IS NOT NULL AND f.n_service_rows >= 5
ORDER BY g.gbm_score DESC LIMIT 1
""")["npi"].iloc[0])
pos_npi = int(q("""
SELECT f.npi FROM provider_features f JOIN scores_gbm g USING (npi)
WHERE f.label = 1 AND f.n_service_rows >= 5
ORDER BY g.gbm_score DESC LIMIT 1
""")["npi"].iloc[0])
print(provider_context(top_npi))

Provider NPI 1578107959: Nurse Practitioner, MA, individual.
Model rank: gradient-boosting score 0.968 (calibrated probability 0.0004), isolation-forest anomaly score 0.460, 6 features beyond 3 peer spreads.
Peer-adjusted z-scores driving the rank (positive = above peers of the same specialty):
  pymt_per_bene_per_risk: z = +16.15  (raw value 1.03e+03)
  pymt_per_bene: z = +15.73  (raw value 1.25e+03)
  srvcs_per_bene: z = +6.70  (raw value 13.3)
  pct_dual: z = +4.28  (raw value 0.636)
  pct_under65: z = +4.16  (raw value 0.543)
Top billed codes in 2024 by Medicare payment (this provider's actual billing):
  90837 (office): Psychotherapy, 1 hour | 51 patients, 716 services, $69,548 (43% of payments)
  99214 (office): Established patient office or other outpatient visit with moderate level of deci | 95 patients, 587 services, $45,320 (28% of payments)
  99215 (office): Established patient office or other outpatient visit with high level of medical  | 55 patients, 323 services, $38,537 

## 5. The chain

**What this cell does:** wires four pieces together with LangChain's pipe syntax (called LCEL,
the LangChain Expression Language):

```
{context, codes, question}  ->  prompt  ->  Claude  ->  string
```

- `context` comes from `provider_context()` (DuckDB).
- `codes` comes from the retriever, run on the investigator's question, so the model gets
  descriptions of whatever codes the question is about, even if the provider does not bill them.
- `question` passes straight through.

The prompt tells the model what it is and what it must not do. In particular it must not say
"fraud." The model has a ranking and some billing facts. It does not have a case.

One rule in the prompt exists because of a failure seen while building this: when a provider
has no published code rows, the model reached for the retrieved reference codes and described
them as the provider's billing. Retrieved context and provider facts have to be labeled as
different things, or the model will blend them. That is the most common way RAG systems go
wrong, and it is fixed in the prompt, not the retriever.

Claude is called through LangChain's `ChatAnthropic` wrapper, which sits on the same Anthropic
SDK Colo uses directly. Model: `claude-opus-5`.

In [6]:
SYSTEM = """You are an analyst supporting a Medicare program-integrity team. You are given
facts about one provider's 2024 Part B billing, their rank under two statistical models, and
reference descriptions of relevant procedure codes. Write a short, plain-English briefing for
an investigator who has five minutes.

Rules:
- The PROVIDER FACTS section is the only source of truth about what this provider billed. The
  REFERENCE CODE DESCRIPTIONS section is general background retrieved for the question; those
  codes may or may not appear in this provider's billing. Never attribute a reference code to
  the provider unless it is listed under PROVIDER FACTS.
- Describe patterns. Never say or imply the provider committed fraud; the models rank, they do not decide.
- Explain each unusual number in one sentence, using the code descriptions to say what the
  services actually are.
- Offer two or three specific, checkable next questions an investigator could pursue.
- If the pattern has an obvious innocent explanation, say so.
- Under 250 words. No bullet-point walls; short paragraphs."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "PROVIDER FACTS\n{context}\n\nREFERENCE CODE DESCRIPTIONS\n{codes}\n\nQUESTION\n{question}"),
])

def format_docs(ds):
    return "\n".join(d.page_content for d in ds)

llm = ChatAnthropic(model="claude-opus-5", max_tokens=1024) if HAVE_KEY else None

chain = (
    {
        "context": RunnableLambda(lambda x: provider_context(x["npi"])),
        "codes": RunnableLambda(lambda x: x["question"]) | retriever | RunnableLambda(format_docs),
        "question": RunnableLambda(lambda x: x["question"]),
    }
    | prompt
    | (llm | StrOutputParser() if HAVE_KEY else RunnableLambda(lambda p: "[no API key] prompt would be:\n\n" + p.to_string()))
)
print("chain built")

chain built


**What this cell does:** runs the chain on the highest-scoring labeled provider. The
`question` is what an investigator would actually ask.

In [7]:
answer = chain.invoke({
    "npi": top_npi,
    "question": "What is unusual about this provider's billing, and what would you check first?",
})
print(answer)

# Briefing: NPI 1578107959 (Nurse Practitioner, Massachusetts)

**What the models saw.** This provider ranks high on the gradient-boosting model (0.968) and shows six features more than three peer spreads from the norm — though the calibrated probability is very low (0.0004), so this is a statistical outlier flag, not a finding.

**The unusual numbers.** Payment per beneficiary is about $1,250 (z = +15.7), and stays high even after risk adjustment (~$1,030, z = +16.2), meaning the spending isn't explained by how sick the panel is. Services per beneficiary average 13.3 (z = +6.7), indicating patients are seen repeatedly rather than once or twice. The panel also skews toward dual-eligible (64%) and under-65 (54%) beneficiaries, both well above peers.

**What's driving it.** Roughly 43% of payments come from 90837, hour-long psychotherapy — 716 sessions across 51 patients, or about 14 sessions per patient. Another 52% comes from moderate- and high-complexity established office visits (992

**What this cell does:** same chain, on the known positive the model ranked highest. Compare
the two briefings. The model was not told which one was excluded.

In [8]:
answer_pos = chain.invoke({
    "npi": pos_npi,
    "question": "Is this billing pattern consistent with legitimate high-volume practice, or does it warrant review?",
})
print(answer_pos)

# Briefing: NPI 1922551399 — Clinical Laboratory, TX

**Bottom line:** The statistical models place this lab low on the priority list (calibrated probability 0.0004), but one feature is extreme enough to justify a quick look.

**What's driving the rank.** Essentially everything rests on a single measure: the same-day repeat ratio is 1.69, about 57 peer standard deviations above other clinical labs — meaning that on average each patient encounter generated roughly 1.7 units of the same code on the same day. It shows up most clearly in 87801 (amplified-probe nucleic acid detection for multiple organisms), where 218 patients produced 721 services, and 87798 (amplified-probe detection for a single organism), with 64 patients and 204 services. The other flagged numbers are mild: average patient age 79, payment per beneficiary $356, and risk-adjusted payment $274 per beneficiary — all within about one standard deviation of peers. No dual-eligible patients were billed.

**Innocent explanation

## 6. A simple agent

The chain above always does the same three lookups in the same order. An **agent** lets the
model decide which lookups to make. The two functions below are the same DuckDB and retriever
calls, exposed as **tools** with a docstring the model reads to decide when to use them.

**What this cell does:** defines the tools, binds them to Claude, and runs a short loop: send
the question, execute whatever tool the model asks for, send the result back, repeat until it
answers. This is the manual tool-use loop from the direct-API days; LangChain's `bind_tools`
just handles the schema plumbing. Agent frameworks (LangGraph, the SDK tool runner) wrap
exactly this loop.

In [9]:
@tool
def lookup_provider(npi: int) -> str:
    """Return billing facts, model scores, and peer-adjusted z-scores for one provider by NPI."""
    return provider_context(int(npi))

@tool
def search_codes(query: str) -> str:
    """Find HCPCS procedure codes whose descriptions match a plain-English query. Use it to
    decode unfamiliar codes or to find which codes cover a type of service."""
    return format_docs(retriever.invoke(query))

TOOLS = {"lookup_provider": lookup_provider, "search_codes": search_codes}

def run_agent(question: str, max_steps: int = 6):
    if not HAVE_KEY:
        print("[no API key] agent not run"); return
    agent_llm = llm.bind_tools(list(TOOLS.values()))
    messages = [("system", "You are a program-integrity analyst. Use the tools to gather facts, "
                           "then answer in under 200 words. Never state that fraud occurred."),
                ("human", question)]
    for step in range(max_steps):
        ai = agent_llm.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            print(ai.content if isinstance(ai.content, str) else ai.content[-1].get("text", ""))
            return
        for call in ai.tool_calls:
            print(f"  [step {step + 1}] tool call: {call['name']}({call['args']})")
            result = TOOLS[call["name"]].invoke(call["args"])
            messages.append({"role": "tool", "content": result, "tool_call_id": call["id"]})
    print("(stopped: step limit)")

run_agent(f"Provider {top_npi} is at the top of our list. Look them up, decode their top two "
          f"codes if you do not recognize them, and tell me whether the pattern is worth a reviewer's time.")

  [step 1] tool call: lookup_provider({'npi': 1578107959})


**Facts.** I recognize all five codes, so no lookup was needed: 90837 is 60-minute individual psychotherapy, 99214/99215 are moderate- and high-complexity established-patient visits, and 90791/90792 are psychiatric diagnostic evaluations. This is a coherent behavioral-health practice profile for a nurse practitioner.

**Why the rank is misleading.** The gradient-boosting *rank* is 0.968 but the *calibrated probability* is 0.0004 — the model is not actually confident. The large z-scores come from benchmarking against all nurse practitioners, most of whom do primary care. A psychiatric NP inherently has a small panel seen repeatedly, so payment-per-beneficiary (z = +16) and services-per-beneficiary (13.3 visits/yr) are expected, not aberrant. Elevated dual-eligible (64%) and under-65 (54%) shares are also characteristic of a Medicare behavioral-health population.

**Workload check.** 716 hour-long sessions plus ~910 office visits is a demanding but achievable full-time year.

**Recommend

## 7. What to carry forward

**The pieces and their names.** Documents, an embedding model, a vector store, a retriever, a
prompt template, a chain, an output parser, tools, an agent loop. Every one of them is
something the Colo project did with hand-written code. LangChain gives them a shared interface
so a vector store or a model can be swapped in one line.

**What retrieval is for here.** Not to find fraud. The models do that. Retrieval turns a row of
z-scores and code numbers into something a human can act on, and it lets an investigator ask
questions in their own words. That is the realistic scope of generative AI in program
integrity today: explanation and triage support, with a person deciding.

**The interview answer to "have you used LangChain":** "I built my earlier RAG system with
direct API calls to understand each piece. For this project I rebuilt the same architecture in
LangChain: a Chroma vector store with a retriever, an LCEL chain that merges structured
context from the database with retrieved code descriptions, and a small tool-calling agent.
The framework saves plumbing; understanding what it wraps is what makes it debuggable."

In [10]:
con.close()